In [ ]:
# auto_stop_fulln.ipynb -- overnight sentinel. Run on EVERY pod before sleep.
# Stops training cleanly (supervisor SIGTERM -> lease self-revokes) and then
# stops THIS RunPod pod (billing ends; /workspace persists) when ANY of:
#   1. FULL-N group fully terminal (normal finish);
#   2. STALL: FULL-N remaining not DECREASED for STALL_HOURS (fleet assumed
#      broken; all sentinels see the same counter -> whole fleet stops);
#   3. LOCAL GRADUATION: this machine trains no FULL-N combo AND none is
#      claimable -> our part is over; stop instead of burning small-n.
# Interrupted combos are free (checkpoints resume; attempts refunded).
#
# SELF-STOP LADDER (pod DNS has been seen broken: 'lookup api.runpod.io on
# 127.0.0.11:53: server misbehaving'): each stop attempt tries runpodctl ->
# GraphQL on api.runpod.io / api.runpod.ai -> REST rest.runpod.io -> a
# DNS-over-HTTPS bypass (resolve via https://1.1.1.1, then raw TLS with SNI,
# no system DNS involved). Attempts repeat for up to 15 minutes.
#
# FIRST TIME: on ONE pod set TEST_SHUTDOWN=True -> dress rehearsal now.
import copy, json, os, re, shutil, signal, socket, ssl, subprocess, sys, time
from pathlib import Path

TEST_SHUTDOWN = False
RUNPOD_API_KEY_OVERRIDE = ""   # pod env RUNPOD_API_KEY is used when set; paste
                               # a console-created key here only if it is not.
REPO = "/workspace/stable-query-latent"
OUT_DIR = "VICReg_review/heads/cloud_full_sweep_a100"
POLL_SECONDS = 120
STALL_HOURS = 4          # no FULL-N progress for this long -> assume broken, stop
GRAD_POLLS = 2           # consecutive polls of local graduation before stopping
STOP_RETRY_MINUTES = 15  # keep retrying the stop ladder this long

# ---- preflight: fail LOUDLY now, not at 5am --------------------------------
pod_id = os.environ.get('RUNPOD_POD_ID')
ctl = shutil.which('runpodctl')
api_key = RUNPOD_API_KEY_OVERRIDE or os.environ.get('RUNPOD_API_KEY', '')
print(f'preflight: RUNPOD_POD_ID={pod_id or "MISSING"}  runpodctl={ctl or "MISSING"}  '
      f'api_key={"set" if api_key else "MISSING"}')
if ctl and api_key:
    r = subprocess.run(['runpodctl', 'config', '--apiKey', api_key],
                       capture_output=True, text=True)
    print(f'preflight: runpodctl config -> {(r.stdout or r.stderr).strip()[:100]}')
if not pod_id or not api_key:
    print('!! preflight FAILED: this pod cannot stop itself reliably. Fix before')
    print('   sleeping (echo $RUNPOD_API_KEY; if empty, create a key in the')
    print('   RunPod console and paste it into RUNPOD_API_KEY_OVERRIDE above).')

_GQL = {'query': 'mutation { podStop(input: {podId: "%s"}) { id desiredStatus } }'}


def _http_post(url, body, headers=None, timeout=30):
    import urllib.request
    data = json.dumps(body).encode('utf-8')
    req = urllib.request.Request(url, data=data, method='POST',
                                 headers={'Content-Type': 'application/json',
                                          **(headers or {})})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return resp.status, json.loads(resp.read().decode('utf-8') or '{}')


def _doh_resolve(name):
    """Resolve A records via Cloudflare DoH at a LITERAL IP -- works when the
    pod's own DNS (127.0.0.11) is broken."""
    import urllib.request
    req = urllib.request.Request(f'https://1.1.1.1/dns-query?name={name}&type=A',
                                 headers={'accept': 'application/dns-json'})
    with urllib.request.urlopen(req, timeout=15) as resp:
        data = json.loads(resp.read().decode('utf-8'))
    return [a['data'] for a in data.get('Answer', []) if a.get('type') == 1]


def _graphql_stop_via_ip(ip, hostname):
    """Raw HTTPS to a resolved IP with proper SNI -- no system DNS at all."""
    body = json.dumps({'query': _GQL['query'] % pod_id}).encode('utf-8')
    head = (f'POST /graphql?api_key={api_key} HTTP/1.1\r\n'
            f'Host: {hostname}\r\nContent-Type: application/json\r\n'
            f'Content-Length: {len(body)}\r\nConnection: close\r\n\r\n').encode()
    ctx = ssl.create_default_context()
    with socket.create_connection((ip, 443), timeout=30) as sock:
        with ctx.wrap_socket(sock, server_hostname=hostname) as tls:
            tls.sendall(head + body)
            resp = b''
            while True:
                chunk = tls.recv(65536)
                if not chunk:
                    break
                resp += chunk
    txt = resp.decode('utf-8', 'ignore')
    payload = txt[txt.find('{'): txt.rfind('}') + 1]
    return json.loads(payload) if payload else {}


def _accepted(out):
    return bool(((out.get('data') or {}).get('podStop') or {}).get('id'))


def try_stop_once():
    if ctl:
        r = subprocess.run(['runpodctl', 'stop', 'pod', pod_id],
                           capture_output=True, text=True)
        print(f'  runpodctl: rc={r.returncode} {(r.stdout or r.stderr).strip()[:150]}',
              flush=True)
        if r.returncode == 0:
            return True
    if not api_key:
        return False
    for hostname in ('api.runpod.io', 'api.runpod.ai'):
        try:
            _st, out = _http_post(f'https://{hostname}/graphql?api_key={api_key}',
                                  {'query': _GQL['query'] % pod_id})
            print(f'  graphql {hostname}: {json.dumps(out)[:150]}', flush=True)
            if _accepted(out):
                return True
        except Exception as exc:
            print(f'  graphql {hostname}: {type(exc).__name__}: {exc}', flush=True)
    try:
        st, out = _http_post(f'https://rest.runpod.io/v1/pods/{pod_id}/stop', {},
                             {'Authorization': f'Bearer {api_key}'})
        print(f'  rest: http {st} {json.dumps(out)[:120]}', flush=True)
        if st in (200, 201):
            return True
    except Exception as exc:
        print(f'  rest: {type(exc).__name__}: {exc}', flush=True)
    try:
        ips = _doh_resolve('api.runpod.io')
        print(f'  doh: api.runpod.io -> {ips}', flush=True)
        for ip in ips[:2]:
            out = _graphql_stop_via_ip(ip, 'api.runpod.io')
            print(f'  raw-tls {ip}: {json.dumps(out)[:150]}', flush=True)
            if _accepted(out):
                return True
    except Exception as exc:
        print(f'  doh/raw-tls: {type(exc).__name__}: {exc}', flush=True)
    return False


def stop_pod():
    deadline = time.time() + STOP_RETRY_MINUTES * 60
    attempt = 0
    while time.time() < deadline:
        attempt += 1
        print(f'sentinel: stop attempt {attempt}', flush=True)
        try:
            if try_stop_once():
                print('sentinel: STOP ACCEPTED -- pod should show Stopped shortly.',
                      flush=True)
                return True
        except Exception as exc:
            print(f'  attempt error: {type(exc).__name__}: {exc}', flush=True)
        time.sleep(30)
    return False


if REPO not in sys.path:
    sys.path.insert(0, REPO)
from VICReg_review.sweep.config import SweepConfig

root = Path(REPO) / OUT_DIR
cfg = SweepConfig.load(f'{REPO}/VICReg_review/sweep/sweep.yaml')
combos = list(cfg.iter_combos())
_counts = [int(c.train_games) for c in combos]
FULL_N = 0 if any(n <= 0 for n in _counts) else max(_counts)
fulln_ids = [c.combo_id for c in combos if int(c.train_games) == FULL_N]
HOST = socket.gethostname()
_m = re.fullmatch(r'Pod_(\d+)', Path.cwd().name)
BUNDLE_BASE = f'VM{_m.group(1)}' if _m else None
print(f'sentinel: {len(fulln_ids)} FULL-N combos; poll={POLL_SECONDS}s; '
      f'stall={STALL_HOURS}h; host={HOST}; bundle={BUNDLE_BASE or "-"}; '
      f'TEST_SHUTDOWN={TEST_SHUTDOWN}')


def _read(p):
    try:
        return json.loads(Path(p).read_text(encoding='utf-8'))
    except Exception:
        return None


def _terminal(cid):
    d = root / cid
    if (d / 'done.json').exists() or (d / 'failed.json').exists():
        return True
    man = _read(d / 'vicreg_review_h5_manifest.json')
    return bool(man) and man.get('status') == 'done'


def fresh_vms():
    now = time.time()
    out = {}
    for f in (root / 'VM_parallel').glob('*.json'):
        rec = _read(f) or {}
        if float(rec.get('expiry', 0) or 0) > now:
            out[str(rec.get('vm') or f.stem)] = rec
    return out


def my_vm_names(vms):
    mine = set()
    fam = re.compile(re.escape(BUNDLE_BASE) + r'(?:_\d+)?$') if BUNDLE_BASE else None
    for name, rec in vms.items():
        if (rec.get('info') or {}).get('host') == HOST:
            mine.add(name)
        elif fam and fam.fullmatch(name):
            mine.add(name)
    return mine


def survey():
    vms = fresh_vms()
    mine = my_vm_names(vms)
    remaining_ids, i_train, claimable = [], False, False
    for cid in fulln_ids:
        if _terminal(cid):
            continue
        remaining_ids.append(cid)
        st = _read(root / cid / 'status.json')
        owner = st.get('vm') if st else None
        if owner in mine:
            i_train = True
        elif owner is None or owner not in vms:
            claimable = True
    return remaining_ids, i_train, claimable


def stop_everything(reason):
    print(f'sentinel: {reason} -- stopping training on this pod', flush=True)
    subprocess.run(['pkill', '-f', 'sweep/supervisor.py'])   # SIGTERM: lease self-revokes
    time.sleep(10)
    subprocess.run(['pkill', '-9', '-f', 'sweep/worker.py'])
    if not (pod_id and stop_pod()):
        print('!! SELF-STOP FAILED after the whole ladder -- pod is idle but '
              'still billing; stop it from the RunPod console.', flush=True)


if TEST_SHUTDOWN:
    stop_everything('TEST_SHUTDOWN dress rehearsal')
    print('test issued: confirm Stopped in the console, restart the pod, re-run '
          'training.ipynb (interruption is free), then deploy with '
          'TEST_SHUTDOWN=False.')
else:
    last_n = None
    last_progress_ts = time.time()
    grad_streak = 0
    while True:
        left, i_train, claimable = survey()
        n = len(left)
        if last_n is None or n < last_n:
            last_n = n
            last_progress_ts = time.time()
        if n == 0:
            stop_everything('FULL-N group fully terminal')
            break
        if time.time() - last_progress_ts > STALL_HOURS * 3600:
            stop_everything(f'STALL: FULL-N remaining stuck at {n} for '
                            f'{STALL_HOURS}h -- assuming a fleet problem')
            break
        if not i_train and not claimable:
            grad_streak += 1
            if grad_streak >= GRAD_POLLS:
                stop_everything(f'LOCAL GRADUATION: this VM trains no FULL-N combo '
                                f'and none is claimable ({n} remain on other VMs)')
                break
        else:
            grad_streak = 0
        stall_min = (time.time() - last_progress_ts) / 60
        print(f'{time.strftime("%H:%M:%S")} FULL-N remaining: {n} '
              f'(training-fulln-here={i_train} claimable={claimable} '
              f'no-progress={stall_min:.0f}m grad-streak={grad_streak})', flush=True)
        time.sleep(POLL_SECONDS)
